In [ ]:
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import transforms

from torchvision.models import (
    mobilenet_v2,
    MobileNet_V2_Weights
)

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    "Using device:",
    device
)

In [ ]:
def find_project_root():

    current = Path.cwd().resolve()

    while current != current.parent:

        if (current / ".git").exists():
            return current

        current = current.parent

    raise FileNotFoundError(
        "Could not find project root."
    )


PROJECT_ROOT = find_project_root()

DATA_ROOT = (
    PROJECT_ROOT
    / "data"
    / "GTSRB"
)

SPLIT_DIR = (
    PROJECT_ROOT
    / "splits"
)

if str(PROJECT_ROOT) not in sys.path:

    sys.path.insert(
        0,
        str(PROJECT_ROOT)
    )

In [ ]:
from Common.dataset import (
    create_dataloaders
)

from Common.training import (
    train_model
)

from Common.evaluation import (
    get_predictions,
    calculate_metrics,
    plot_training_curves,
    plot_confusion_matrix,
    print_classification_report,
    total_parameters,
    trainable_parameters,
    save_model_and_get_size,
    measure_inference_time
)

In [ ]:
train_df = pd.read_csv(
    SPLIT_DIR
    / "train_split.csv"
)

val_df = pd.read_csv(
    SPLIT_DIR
    / "val_split.csv"
)

test_df = pd.read_csv(
    SPLIT_DIR
    / "test_split.csv"
)

ImageNet transforms

In [ ]:
imagenet_train_transform = transforms.Compose([
    transforms.Resize(
        (64, 64)
    ),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),

    transforms.RandomRotation(
        degrees=8
    ),

    transforms.RandomAffine(
        degrees=0,
        translate=(0.05, 0.05),
        scale=(0.95, 1.05)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


imagenet_eval_transform = transforms.Compose([
    transforms.Resize(
        (64, 64)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])

MobileNet DataLoaders

In [ ]:
train_loader, val_loader, test_loader = (
    create_dataloaders(
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        data_root=DATA_ROOT,
        batch_size=32,
        num_workers=0,
        train_transform_override=imagenet_train_transform,
        eval_transform_override=imagenet_eval_transform,
        seed=SEED
    )
)

Load MobileNet V2

In [ ]:
weights = (
    MobileNet_V2_Weights.DEFAULT
)

mobile_model = mobilenet_v2(
    weights=weights
)

for parameter in mobile_model.parameters():
    parameter.requires_grad = False

In [ ]:
input_features = (
    mobile_model
    .classifier[1]
    .in_features
)

mobile_model.classifier[1] = nn.Linear(
    input_features,
    43
)

mobile_model = mobile_model.to(
    device
)

Check ---

In [ ]:
print(mobile_model)

print(
    "Total parameters:",
    total_parameters(
        mobile_model
    )
)

print(
    "Trainable parameters:",
    trainable_parameters(
        mobile_model
    )
)

Mn training

In [ ]:
criterion = nn.CrossEntropyLoss()

mobile_optimizer = optim.Adam(
    filter(
        lambda parameter:
        parameter.requires_grad,
        mobile_model.parameters()
    ),
    lr=1e-3
)

In [ ]:
mobile_model, mobile_head_history = (
    train_model(
        model=mobile_model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=mobile_optimizer,
        criterion=criterion,
        epochs=10,
        device=device
    )
)

In [ ]:
for parameter in (
    mobile_model
    .features[-3:]
    .parameters()
):
    parameter.requires_grad = True

In [ ]:
mobile_optimizer_ft = optim.Adam(
    filter(
        lambda parameter:
        parameter.requires_grad,
        mobile_model.parameters()
    ),
    lr=1e-5
)

In [ ]:
mobile_model, mobile_ft_history = (
    train_model(
        model=mobile_model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=mobile_optimizer_ft,
        criterion=criterion,
        epochs=20,
        device=device
    )
)

Plots

In [ ]:
mobile_history = {}

for key in mobile_head_history.keys():

    mobile_history[key] = (
        mobile_head_history[key]
        + mobile_ft_history[key]
    )

In [ ]:
FIGURES_DIR = (
    PROJECT_ROOT
    / "results"
    / "figures"
)

plot_training_curves(
    mobile_history,
    "MobileNetV2",
    FIGURES_DIR
)

Evaluation

In [ ]:
y_true_mobile, y_pred_mobile = (
    get_predictions(
        mobile_model,
        test_loader,
        device
    )
)

mobile_metrics = (
    calculate_metrics(
        y_true_mobile,
        y_pred_mobile
    )
)

print(
    mobile_metrics
)

In [ ]:
plot_confusion_matrix(
    y_true_mobile,
    y_pred_mobile,
    "MobileNetV2",
    FIGURES_DIR
)

print_classification_report(
    y_true_mobile,
    y_pred_mobile
)

size and time

In [ ]:
mobile_total_params = total_parameters(
    mobile_model
)

mobile_model_path = (
    PROJECT_ROOT
    / "checkpoints"
    / "mobilenetv2_final.pth"
)

mobile_size_kb = (
    save_model_and_get_size(
        mobile_model,
        mobile_model_path
    )
)

mobile_size_mb = (
    mobile_size_kb
    / 1024
)

mobile_inference = (
    measure_inference_time(
        mobile_model,
        test_loader,
        device
    )
)

print(
    "Parameters:",
    mobile_total_params
)

print(
    "Model size:",
    mobile_size_mb,
    "MB"
)

print(
    "Inference:",
    mobile_inference,
    "ms/image"
)

In [ ]:
mobile_results = {
    "Model": "MobileNetV2",
    "Parameters": mobile_total_params,
    "Model Size (MB)": mobile_size_mb,
    "Inference Time (ms/image)": mobile_inference,
    "Test Accuracy": mobile_metrics["Test Accuracy"],
    "Macro Precision": mobile_metrics["Macro Precision"],
    "Macro Recall": mobile_metrics["Macro Recall"]
}

pd.DataFrame(
    [mobile_results]
).to_csv(
    PROJECT_ROOT
    / "results"
    / "metrics"
    / "mobilenetv2_metrics.csv",
    index=False
)